# Local RAG Pipeline with HuggingFace Embeddings (Python 3.12)

This notebook demonstrates how to:

- ~~Read local `.txt` and `.pdf` files~~
  - ~~This is good enough for 'playing' but production would need many more examples of files AND longer files.~~
  - ~~A single service would not be enough``~~
- ~~Chunk text for embedding~~
- Generate embeddings using a local HuggingFace model
- Save and load vector index locally
- ~~Perform cosine similarity search without any external services~~

In [ ]:
%pip install --upgrade pip

%pip install -q -r requirements.txt

In [1]:
from config import settings
import numpy as np

print("Numpy ", np.__version__)

Numpy  1.26.4


## Transformer

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


In [3]:
print("Loading SentenceTransformer model..." + settings.SENTENCE_TRANSFORMER_MODEL)

chunks = []
current_chunk = {}

with open(settings.CHUNKED_FILE, "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if line.startswith("__META__FILE__NAME:"):
            if current_chunk:  # Save the previous one
                chunks.append(current_chunk)
                current_chunk = {}
            current_chunk["file"] = line.replace("__META__FILE__NAME: ", "")
        elif line.startswith("__ID__:"):
            current_chunk["id"] = line.replace("__ID__: ", "")
        elif line.startswith("__META__CHUNK:"):
            current_chunk["chunk"] = line.replace("__META__CHUNK: ", "")
        elif line.startswith("__META__DATE:"):
            current_chunk["date"] = line.replace("__META__DATE: ", "")
        elif line.startswith("__CONTENT__:"):
            current_chunk["text"] = line.replace("__CONTENT__: ", "")

# Append final one
if current_chunk:
    chunks.append(current_chunk)

# Now embed
texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(texts, show_progress_bar=True)

Loading SentenceTransformer model...all-MiniLM-L6-v2


FileNotFoundError: [Errno 2] No such file or directory: 'chunked_data.txt'

In [ ]:
print(embeddings.shape)

In [ ]:
print(embeddings[0])

In [ ]:
import pandas as pd
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

df = pd.DataFrame({"text": chunks, "embedding": embeddings.tolist()})
df.to_parquet(settings.VECTOR_STORE, index=False)
print("Saved to vector_index_local.parquet")